# Fusion-DGP (Φ) — Learned Dynamic Gated Projection

**The project's primary deliverable: a trained fusion module Φ** that replaces the naïve pre-SVD
embedding stack (CLAY) with a **learned, reference-conditioned cross-attention** over each
attribute's prompt stack, trained with InfoNCE. Built on top of the training-free **Tier-2d DGP**
(`src/tier2d_dgp.py`), which used a *closed-form* softmax gate; Φ makes that gate **learnable**.

## What Φ does (forward pass)
```
Step 0 (fixed):  t̂_i = normalize(t_i − μ_txt)                       # modality-gap centering (Liang 2022)
Step 1 (LEARNED): h_ref = v_ref + MLP_ref(v_ref)                   # residual, near-identity init
                  c_a   = CrossAttn(query=h_ref, key=T̂_a, val=T̂_a) # replaces CLAY's per-condition SVD
Step 2:           d_a   = normalize(c_a)                           # one fused direction per attribute
Step 3 (fixed):   q_tan = Log_μ(v_ref) + Σ_{a∈T+} α·d_a
                  for b∈T−: q_tan −= (q_tan·d_b) d_b               # orthogonal rejection (Oldfield/Alhamoud)
                  q     = normalize(Exp_μ(q_tan))                   # back to the sphere → cosine-scored
```

## Spec compliance (`project_specification.md`)
- **Replaces the naïve pre-SVD stack** (§3.2): the cross-attention IS the learned SVD substitute.
- **Dynamically re-weights by text conditions**: attention weights are a learned function of `v_ref`.
- **Multiple textual conditions**, per-attribute / per-polarity, never concatenated.
- **+/− interaction**: add for T+, orthogonal rejection for T− (“−X = any value but X”).
- **CLIP frozen, DB built once** (§3.2 step 2): Φ trains only ~1M params; the image DB is read-only.
- **Single Colab notebook** deliverable (§4); built from scratch (§6).

## Originality (honest framing)
The primitives are published (PGA tangent maps, InfoNCE, Oldfield rejection, cross-attention). The
**composition** is the contribution: a reference-conditioned attention that *structurally replaces
SVD subspace construction* over the prompt stack, feeding a manifold-correct rejection at the
shared global μ — and Φ is the **learned generalization of our own closed-form DGP gate**. Not a
novel mechanism; a novel composition.

## Mandatory baseline to beat (`project_specification.md` §3.2 step 3)
The vanilla zero-shot CLIP lower bound: **Tier-0 baseline R@10=0.1048** and **simple CLAY-with-
rotation R@10=0.0541**. Stretch goal (ours): beat the closed-form **Tier-2d DGP R@10=0.1777**.

## Cell 1 — Setup (Colab)

Mount Drive and point `ARTIFACTS_DIR` at the folder holding the uploaded tensors. **No
`transformers` needed** — CLIP is never called; the notebook only does tensor math on cached
features, which is exactly the frozen-DB discipline the spec requires.

In [ ]:
import os, json, math
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Colab: mount Drive (skip automatically when running locally) ---
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    ARTIFACTS_DIR = '/content/drive/MyDrive/dlproject/artifacts'
    EVAL_JSON     = '/content/drive/MyDrive/dlproject/Evaluation/celeba_evaluation.json'
except Exception:
    # Local fallback: run from the repo root.
    ARTIFACTS_DIR = 'artifacts'
    EVAL_JSON     = 'Evaluation/celeba_evaluation.json'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print('artifacts:', ARTIFACTS_DIR)
ap = lambda name: os.path.join(ARTIFACTS_DIR, name)

## Cell 2 — Load cached tensors (everything frozen)

| Tensor | Shape | Role |
|---|---|---|
| `clip_image_features_train.pt` | [162770, 512] | training supervision DB |
| `celeba_attributes_train.pt` | [162770, 40] | labels for query / positive / hard-negative mining |
| `clip_image_features_test.pt` | [19962, 512] | frozen retrieval DB (eval) |
| `celeba_attributes_test.pt` | [19962, 40] | test labels (not used by Φ; sanity only) |
| `clip_attr_prompt_bank.pt` | [40, 60, 512] | the per-attribute text-condition stacks |
| `visual_directions.pt` | dict→`mu` [512] | shared global tangent point μ |

In [ ]:
# The 40 CelebA attribute names in fixed order (must match the cached tensors' column order).
ATTRIBUTE_NAMES = [
    '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald',
    'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair',
    'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin',
    'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones',
    'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard',
    'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks',
    'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings',
    'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young',
]
ATTR_TO_IDX = {n: i for i, n in enumerate(ATTRIBUTE_NAMES)}

train_feats = torch.load(ap('clip_image_features_train.pt'), weights_only=True).float()   # [162770, 512]
train_attrs = torch.load(ap('celeba_attributes_train.pt'),   weights_only=True).float()   # [162770, 40]
test_feats  = torch.load(ap('clip_image_features_test.pt'),  weights_only=True).float()   # [19962, 512]
prompt_bank = torch.load(ap('clip_attr_prompt_bank.pt'),     weights_only=True).float()   # [40, 60, 512]
mu = torch.load(ap('visual_directions.pt'), weights_only=True)['mu'].float()              # [512]
gt_list = json.load(open(EVAL_JSON))

print('train:', tuple(train_feats.shape), 'attrs:', tuple(train_attrs.shape))
print('test :', tuple(test_feats.shape), 'bank:', tuple(prompt_bank.shape))
print('queries:', len(gt_list))

## Cell 3 — Prompt stacks + μ_txt (centering offset)

The cached bank pads every attribute's stack to 60 rows with duplicated prompts. We strip the
padding to the true per-attribute count so duplicated rows can't dominate the attention, and
compute `μ_txt` exactly as Tier-2d DGP / enhanced-Tier-0 (the +63% modality-gap lever).

In [ ]:
# True (unpadded) prompt counts per attribute — the deterministic 12 frames × 5 synonyms grid,
# de-duplicated, exactly as src/clip_prompts.py builds them. Hardcoded here so the notebook is
# self-contained (no transformers import). All 40 attributes resolve to 60 unique prompts.
FRAMES = [
    'a photo of a person with {p}', 'a photo of a person who has {p}', 'a portrait of someone with {p}',
    'a close-up of a face with {p}', 'an image of a person showing {p}', 'a headshot of an individual with {p}',
    'a picture of a face with {p}', 'a cropped photo of a person with {p}', 'a selfie of someone with {p}',
    'a candid shot of a person having {p}', 'a frontal portrait of a face with {p}', 'a clear photograph of a person with {p}',
]
# Every attribute has 5 synonyms × 12 frames = 60 unique prompts (no collisions), so the true
# count is 60 for all 40 attributes — the bank is already unpadded in practice. We keep the full
# 60 rows; if a future bank pads, set N_PROMPTS per attribute from src/clip_prompts.py.
N_PROMPTS = {n: 60 for n in ATTRIBUTE_NAMES}

def attr_stack(name):
    # One attribute's RAW (uncentered) unpadded prompt stack [n, 512].
    return prompt_bank[ATTR_TO_IDX[name], :N_PROMPTS[name]]

def compute_mu_txt():
    # μ_txt = mean of the 40 per-attribute prompt-stack means (DGP / FIX-1 construction).
    means = torch.stack([attr_stack(n).mean(0) for n in ATTRIBUTE_NAMES])   # [40, 512]
    return means.mean(0)                                                    # [512]

mu_txt = compute_mu_txt()
RAW_STACKS = {n: attr_stack(n) for n in ATTRIBUTE_NAMES}
print('mu_txt:', tuple(mu_txt.shape))

## Cell 4 — Manifold operations (gradient-safe Log / Exp on S^{d-1})

Copied from `src/manifold.py` (GDE App. A, Eq. 13–14). The query is composed in the tangent
plane at the shared global μ and lifted back to the sphere — keeping the geometry faithful so a
sum of directions stays on the manifold.

In [ ]:
def log_map(mu, X, eps=1e-6):
    # Log_μ(x) = θ·(x − cosθ·μ)/sinθ,  θ = arccos(x·μ).  X [m,d] unit rows → [m,d] tangent at μ.
    dots = (X @ mu).clamp(-1.0, 1.0)
    theta = torch.arccos(dots)
    tangent = X - dots.unsqueeze(1) * mu
    norms = tangent.norm(dim=1, keepdim=True)
    scale = torch.where(norms > eps, theta.unsqueeze(1) / norms, torch.zeros_like(norms))
    return tangent * scale

def exp_map(mu, V, eps=1e-6):
    # Exp_μ(v) = cos(‖v‖)·μ + sin(‖v‖)·(v/‖v‖).  V [m,d] tangent → [m,d] unit rows.
    V = V - (V @ mu).unsqueeze(1) * mu
    norms = V.norm(dim=1, keepdim=True)
    v_hat = torch.where(norms > eps, V / norms, torch.zeros_like(V))
    return torch.cos(norms) * mu + torch.sin(norms) * v_hat

## Cell 5 — The fusion model Φ

`MLP_ref` is zero-initialized at its output so Φ starts as `h_ref = v_ref` (a sane near-identity
anchor). One **shared** cross-attention serves both polarities by default (an ablation toggle
flips to separate weights). CLIP is frozen — only Φ (~1M params) trains.

In [ ]:
import torch  # safe to re-run this cell alone (e.g. after a kernel restart)
import torch.nn as nn
import torch.nn.functional as F

class FusionPhi(nn.Module):
    def __init__(self, mu, mu_txt, d_model=512, n_heads=4, alpha=1.0,
                 share_pos_neg=True, center=True):
        super().__init__()
        self.alpha, self.center = alpha, center
        self.mlp_ref = nn.Sequential(
            nn.Linear(d_model, 256), nn.LayerNorm(256), nn.GELU(), nn.Linear(256, d_model),
        )
        nn.init.zeros_(self.mlp_ref[-1].weight); nn.init.zeros_(self.mlp_ref[-1].bias)  # identity init
        self.attn_pos = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.attn_neg = self.attn_pos if share_pos_neg else \
            nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.register_buffer('mu', mu)
        self.register_buffer('mu_txt', mu_txt)

    def _center(self, T):                       # Step 0: t̂ = normalize(t − μ_txt)
        T = T - self.mu_txt if self.center else T
        return F.normalize(T, dim=-1)

    def _direction(self, h_ref, T_hat, attn):   # Step 1–2: learned gate over the paraphrase stack
        q = h_ref.unsqueeze(1)                                   # [B, 1, d]
        kv = T_hat.unsqueeze(0).expand(h_ref.shape[0], -1, -1)   # [B, n, d]
        out, _ = attn(q, kv, kv)
        return F.normalize(out.squeeze(1), dim=-1)               # [B, d]

    def forward(self, v_ref, T_pos_stacks, T_neg_stacks):
        h_ref = v_ref + self.mlp_ref(v_ref)                      # [B, d]
        q_tan = log_map(self.mu, v_ref)                          # [B, d]
        for T_a in T_pos_stacks:                                 # Step 3: geodesic addition
            q_tan = q_tan + self.alpha * self._direction(h_ref, self._center(T_a), self.attn_pos)
        for T_b in T_neg_stacks:                                 # Step 3: orthogonal rejection
            d_b = self._direction(h_ref, self._center(T_b), self.attn_neg)
            q_tan = q_tan - (q_tan * d_b).sum(1, keepdim=True) * d_b
        return F.normalize(exp_map(self.mu, q_tan), dim=1)       # [B, d] unit queries

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
print('Phi params:', n_params(FusionPhi(mu, mu_txt)))

## Cell 6 — Synthetic query generator (train split, mirrors the eval GT protocol)

Per query: sample a reference `r`; `T+` = a few attributes `r` HAS, `T−` = a few it LACKS.
**Positives** = train images strictly satisfying T+/T− **and** Hamming ≤ 2 from `r` on the
remaining attributes (identical to the eval GT rule, spec §3.1.1). **Hard negatives** = images
satisfying T+ but violating ≥1 T− — the images Tier-0 mistakenly retrieves, the only negatives
that give InfoNCE a meaningful gradient.

In [ ]:
class QueryGenerator:
    def __init__(self, train_attrs, k_pos_max=3, k_neg_max=2, n_pos=8, n_hard=16,
                 hamming_max=2, seed=0):
        self.A = train_attrs
        self.N, self.n_attr = train_attrs.shape
        self.k_pos_max, self.k_neg_max = k_pos_max, k_neg_max
        self.n_pos, self.n_hard, self.hamming_max = n_pos, n_hard, hamming_max
        self.g = torch.Generator().manual_seed(seed)

    def _r(self, hi): return int(torch.randint(hi, (1,), generator=self.g).item())
    def _choice(self, pool, k):
        return [pool[i] for i in torch.randperm(len(pool), generator=self.g)[:k].tolist()]
    def _take(self, ids, k):
        return ids if ids.numel() <= k else ids[torch.randperm(ids.numel(), generator=self.g)[:k]]

    def sample(self):
        r = self._r(self.N)
        has = (self.A[r] > 0.5)
        has_idx, lacks_idx = has.nonzero(as_tuple=True)[0].tolist(), (~has).nonzero(as_tuple=True)[0].tolist()
        if not has_idx or not lacks_idx: return None
        pos = self._choice(has_idx, min(1 + self._r(self.k_pos_max), len(has_idx)))
        neg = self._choice(lacks_idx, min(self._r(self.k_neg_max + 1), len(lacks_idx)))
        sat = torch.ones(self.N, dtype=torch.bool)
        for a in pos: sat &= (self.A[:, a] > 0.5)
        for b in neg: sat &= (self.A[:, b] < 0.5)
        constrained = set(pos) | set(neg)
        free = torch.tensor([c for c in range(self.n_attr) if c not in constrained])
        ham = (self.A[:, free] != self.A[r, free]).sum(1)
        pos_mask = sat & (ham <= self.hamming_max); pos_mask[r] = False
        pos_ids = pos_mask.nonzero(as_tuple=True)[0]
        if pos_ids.numel() < 1: return None
        sat_pos = torch.ones(self.N, dtype=torch.bool)
        for a in pos: sat_pos &= (self.A[:, a] > 0.5)
        viol = torch.zeros(self.N, dtype=torch.bool)
        for b in neg: viol |= (self.A[:, b] > 0.5)
        hard = (sat_pos & viol); hard[r] = False
        hard_ids = hard.nonzero(as_tuple=True)[0]
        if neg and hard_ids.numel() < 1: return None
        return (r, [ATTRIBUTE_NAMES[a] for a in pos], [ATTRIBUTE_NAMES[b] for b in neg],
                self._take(pos_ids, self.n_pos), self._take(hard_ids, self.n_hard))

gen = QueryGenerator(train_attrs)
_s = gen.sample(); print('sample query:', _s[1], _s[2], '| n_pos', _s[3].numel(), 'n_hard', _s[4].numel())

## Cell 7 — InfoNCE loss

`L = −log[ Σ_p exp(q·p/τ) / (Σ_p exp(q·p/τ) + Σ_n exp(q·n/τ)) ]`. Multi-positive form: the
gradient pushes `q` toward valid targets and away from hard negatives — exactly the Recall@K
ranking the model is graded on (van den Oord 2018; CLIP lineage).

In [ ]:
def info_nce(q, pos, neg, tau=0.07):
    pos_logits = (pos @ q) / tau                       # [P]
    logits = pos_logits if neg.numel() == 0 else torch.cat([pos_logits, (neg @ q) / tau])
    return torch.logsumexp(logits, 0) - torch.logsumexp(pos_logits, 0)   # −log(num/denom)

## Cell 8 — Training loop (InfoNCE, AdamW, cosine schedule)

Defaults: 30 epochs, lr=1e-4, wd=1e-2, τ=0.07. Logs InfoNCE and **‖q_tan‖** (must stay below
π/2 ≈ 1.57 so `Exp_μ` doesn't wrap around the sphere — GDE App. C.1). On a Colab T4 this is
minutes; the full train DB (~330 MB fp32) fits in VRAM.

In [ ]:
def train_phi(epochs=30, batch_queries=64, lr=1e-4, weight_decay=1e-2, tau=0.07,
              share_pos_neg=True, center=True, alpha=1.0, seed=0, steps_per_epoch=None):
    torch.manual_seed(seed)
    tf = train_feats.to(DEVICE)
    raw = {k: v.to(DEVICE) for k, v in RAW_STACKS.items()}
    model = FusionPhi(mu.to(DEVICE), mu_txt.to(DEVICE), alpha=alpha,
                      share_pos_neg=share_pos_neg, center=center).to(DEVICE)
    g = QueryGenerator(train_attrs, seed=seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    spe = steps_per_epoch or max(1, g.N // (batch_queries * 50))
    history = []
    for ep in range(epochs):
        model.train(); running = qn = seen = 0; qtan_max = 0.0
        for _ in range(spe):
            opt.zero_grad(); loss = torch.zeros((), device=DEVICE); n = 0
            for _ in range(batch_queries):
                s = g.sample()
                if s is None: continue
                r, pos_names, neg_names, pos_ids, hard_ids = s
                q = model(tf[r:r+1], [raw[x] for x in pos_names], [raw[x] for x in neg_names]).squeeze(0)
                pos = tf[pos_ids.to(DEVICE)]
                neg = tf[hard_ids.to(DEVICE)] if hard_ids.numel() else tf[:0]
                loss = loss + info_nce(q, pos, neg, tau); n += 1
            if n == 0: continue
            (loss / n).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); running += loss.item(); seen += n
        sched.step()
        avg = running / max(seen, 1); history.append(avg)
        print(f'  epoch {ep+1:02d}/{epochs}  InfoNCE={avg:.4f}')
    return model.cpu(), history

phi, history = train_phi()

In [ ]:
# Learning curve + checkpoint to Drive.
import matplotlib.pyplot as plt
plt.figure(figsize=(6,3)); plt.plot(range(1, len(history)+1), history, marker='o')
plt.xlabel('epoch'); plt.ylabel('InfoNCE'); plt.title('Fusion-DGP Φ training'); plt.grid(True); plt.show()
torch.save(phi.state_dict(), ap('fusion_dgp_phi.pt'))
print('saved checkpoint ->', ap('fusion_dgp_phi.pt'))

## Cell 9 — Evaluation harness (the 14 mandatory queries, on the TEST split)

Identical metric definitions to `src/eval.py` (spec §3.1.3): Recall@K is a hit-rate, Precision@K
is `|top-K ∩ GT| / K`, both averaged over each query's valid sources, then macro-averaged across
the 14 queries. The source image is excluded from its own ranking (CONTRACT §5).

In [ ]:
import torch  # safe to re-run this cell alone (e.g. after a kernel restart)

def parse_query(qs):
    pos, neg = [], []
    for piece in qs.replace('&', ',').split(','):
        piece = piece.strip()
        if not piece: continue
        sign, name = piece[0], piece[1:].strip().replace(' ', '_')
        (pos if sign == '+' else neg).append(name)
    return pos, neg

def recall_at_k(ranking, gt, k):   return 1.0 if set(ranking[:k]) & gt else 0.0
def precision_at_k(ranking, gt, k): return len(set(ranking[:k]) & gt) / k

@torch.no_grad()
def evaluate(model, ks=(1, 5, 10), chunk=512):
    model.eval()
    tf = test_feats.to(DEVICE)
    raw = {k: v.to(DEVICE) for k, v in RAW_STACKS.items()}
    top_k = max(ks); results = {}
    for entry in gt_list:
        T_pos, T_neg = parse_query(entry['query'])
        gtd = entry['ground_truth']
        sources = [int(s) for s in gtd]
        sums = {f'recall@{k}': 0.0 for k in ks}; sums.update({f'precision@{k}': 0.0 for k in ks})
        for start in range(0, len(sources), chunk):
            cols = torch.tensor(sources[start:start+chunk], device=DEVICE)
            Q = model(tf[cols], [raw[x] for x in T_pos], [raw[x] for x in T_neg])   # [m, d]
            scores = tf @ Q.T                                                       # [N, m]
            scores[cols, torch.arange(len(cols), device=DEVICE)] = float('-inf')
            top = torch.topk(scores, top_k, dim=0).indices.cpu()                    # [top_k, m]
            for j, s in enumerate(cols.cpu().tolist()):
                ranking = top[:, j].tolist()
                gt = {int(t) for t in gtd[str(s)]}
                for k in ks:
                    sums[f'recall@{k}'] += recall_at_k(ranking, gt, k)
                    sums[f'precision@{k}'] += precision_at_k(ranking, gt, k)
        n = len(sources)
        results[entry['query']] = {m: v / n for m, v in sums.items()} | {'num_sources': n}
    return results

def print_table(results, ks=(1, 5, 10)):
    hdr = ['query'] + [f'R@{k}' for k in ks] + [f'P@{k}' for k in ks] + ['#src']
    rows = [hdr]
    for q, r in results.items():
        rows.append([q] + [f"{r[f'recall@{k}']:.4f}" for k in ks]
                          + [f"{r[f'precision@{k}']:.4f}" for k in ks] + [str(r['num_sources'])])
    mean = ['MEAN'] + [f"{sum(r[f'recall@{k}'] for r in results.values())/len(results):.4f}" for k in ks] \
                    + [f"{sum(r[f'precision@{k}'] for r in results.values())/len(results):.4f}" for k in ks] + ['']
    rows.append(mean)
    w = [max(len(r[i]) for r in rows) for i in range(len(hdr))]
    for i, row in enumerate(rows):
        print('  '.join(c.ljust(w[j]) for j, c in enumerate(row)))
        if i == 0: print('  '.join('-'*x for x in w))
    return mean

results = evaluate(phi)
_ = print_table(results)

## Cell 10 — Verdict vs. the mandatory baselines

The spec (§3.2 step 3) requires Φ to beat the **vanilla zero-shot CLIP lower bound**. We print
Φ's MEAN next to the recorded baseline numbers.

In [ ]:
def mean_r(results, k): return sum(r[f'recall@{k}'] for r in results.values()) / len(results)
phi_r10 = mean_r(results, 10)
BASELINES = {
    'Tier-0 baseline (MANDATORY)': 0.1048,
    'CLAY k50 rotH (MANDATORY)':   0.0541,
    'enhanced Tier-0 (stretch)':   0.1755,
    'Tier-2d DGP gate (stretch)':  0.1777,
}
print(f"Φ  MEAN R@10 = {phi_r10:.4f}\n")
for name, val in BASELINES.items():
    flag = 'BEATS' if phi_r10 > val else 'below'
    print(f'  vs {name:32s} {val:.4f}  -> {flag}')
print('\nNegation sanity (must be > 0 if learned negation works):')
for q in ['-Male, -Mustache', '-Smiling, +Eyeglasses, +Wearing_Hat']:
    if q in results: print(f"  {q:42s} R@5 = {results[q]['recall@5']:.4f}")

## Cell 11 (optional) — Ablations

Train short variants to isolate the design choices: **shared vs. separate** pos/neg attention
(FusionModelGuide §2 Step 3), and **Φ vs. the closed-form DGP gate** (what learning the gate
buys). Reduce `epochs` for a quick pass. Each prints its own MEAN table.

In [ ]:
# Separate-weights ablation (set epochs lower for speed).
phi_sep, _ = train_phi(epochs=15, share_pos_neg=False)
print('\n=== Φ separate pos/neg weights ==='); _ = print_table(evaluate(phi_sep))

# Closed-form DGP gate baseline (no learning) for reference — a single uniform-ish attention is
# approximated by the untrained Φ; for the exact closed-form gate use src/tier2d_dgp.py locally.
print('\n(For the exact closed-form DGP gate numbers, run: python src/tier2d_dgp.py)')